# Bayesian Inference: Probabilistic Programming and Belief Updating

## 1. Introduction and Overview

Bayesian Inference is a paradigm of statistical reasoning where probabilities represent degrees of belief rather than long-run frequencies. Instead of treating parameters as fixed and data as random (the Frequentist view), Bayesian inference treats the observed data as fixed and parameters as random variables described by probability distributions.

In this notebook, we will explore the core mechanics of Bayesian Inference. We will start with analytical solutions using Conjugate Priors, move into computational approximations using Markov Chain Monte Carlo (MCMC), and finally apply these concepts to real-world scenarios like A/B Testing and multi-armed bandits.

In [ ]:
# Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings

# Configure pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Configure visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

# Ignore harmless warnings
warnings.filterwarnings('ignore')

# Set global random seed for complete reproducibility
np.random.seed(42)

print("Environment initialized successfully. Random seed set to 42.")

## 2. Core Concept 1: The Bayesian Engine (Prior, Likelihood, Posterior)

Bayes Theorem tells us how to update our beliefs given new evidence:

P(theta | Data) = P(Data | theta) * P(theta) / P(Data)

- P(theta | Data) is the Posterior: Our updated belief about the parameter after seeing data.
- P(Data | theta) is the Likelihood: How probable the data is, given a specific parameter value.
- P(theta) is the Prior: Our initial belief about the parameter before seeing any data.
- P(Data) is the Evidence (Marginal Likelihood): A normalizing constant to ensure probabilities sum to 1.

Because P(Data) is a constant, we often drop it and state:
Posterior is proportional to Likelihood * Prior

In [ ]:
# Data Creation: Synthetic Data for an Advertising Campaign
# We want to estimate the Click-Through Rate (CTR) of a new ad.

n_impressions = 100
true_ctr = 0.12  # 12 percent true underlying CTR

# Generate synthetic clicks using a Binomial distribution
observed_clicks = np.random.binomial(n=n_impressions, p=true_ctr)

print("--- Campaign Results ---")
print(f"Total Impressions (N): {n_impressions}")
print(f"Observed Clicks (k): {observed_clicks}")
print(f"Raw Empirical CTR: {observed_clicks / n_impressions:.3f}")

## 3. Core Concept 2: Analytical Solution via Conjugate Priors

Before resorting to complex numerical approximations, we can solve Bayes Theorem exactly using Conjugate Priors.
A prior is 'conjugate' to a likelihood if the resulting posterior belongs to the same probability distribution family as the prior.

For a Binomial likelihood (like our ad clicks), the conjugate prior is the Beta distribution.
- Prior: Beta(alpha_prior, beta_prior)
- Likelihood: Binomial(k successes, n total)
- Posterior: Beta(alpha_prior + k, beta_prior + n - k)

In [ ]:
# Step 1: Define our Prior belief
# We believe from historical ads that the CTR is usually around 5%, rarely above 20%.
# We can model this with a Beta(2, 38) distribution. Mean = 2 / (2 + 38) = 0.05
alpha_prior = 2
beta_prior = 38

print("--- Prior Parameters ---")
print(f"Prior Alpha (pseudo-clicks): {alpha_prior}")
print(f"Prior Beta (pseudo-ignores): {beta_prior}")
print(f"Expected Prior CTR: {alpha_prior / (alpha_prior + beta_prior):.3f}")

# Step 2: Compute Posterior Parameters analytically using conjugacy
alpha_post = alpha_prior + observed_clicks
beta_post = beta_prior + n_impressions - observed_clicks

print("\n--- Posterior Parameters ---")
print(f"Posterior Alpha: {alpha_post}")
print(f"Posterior Beta: {beta_post}")
print(f"Expected Posterior CTR: {alpha_post / (alpha_post + beta_post):.3f}")

## 4. Visualizing the Conjugate Prior Update

Let us visualize how the Prior belief and the Likelihood (Data) combine to form the Posterior belief. We will compute the continuous probability density functions (PDFs) for the parameter space p between 0 and 1.

In [ ]:
# Define the parameter space (possible CTRs from 0 to 1)
x_ctr = np.linspace(0.001, 0.4, 500) # Zooming in on 0 to 40 percent

# Compute Prior PDF
prior_pdf = stats.beta.pdf(x_ctr, alpha_prior, beta_prior)

# Compute Likelihood (scaled for visualization purposes)
# Likelihood is proportional to p^k * (1-p)^(n-k)
likelihood = (x_ctr**observed_clicks) * ((1 - x_ctr)**(n_impressions - observed_clicks))
likelihood_scaled = (likelihood / np.max(likelihood)) * np.max(prior_pdf)

# Compute Posterior PDF
posterior_pdf = stats.beta.pdf(x_ctr, alpha_post, beta_post)

plt.figure(figsize=(12, 6))
plt.plot(x_ctr, prior_pdf, label=f'Prior: Beta({alpha_prior}, {beta_prior})', color='gray', linestyle='--', linewidth=2)
plt.plot(x_ctr, likelihood_scaled, label=f'Likelihood (Scaled): Data', color='blue', alpha=0.4, linewidth=2)
plt.plot(x_ctr, posterior_pdf, label=f'Posterior: Beta({alpha_post}, {beta_post})', color='purple', linewidth=3)

plt.axvline(true_ctr, color='red', linestyle=':', label='True Underlying CTR')
plt.axvline(observed_clicks / n_impressions, color='blue', linestyle=':', label='Empirical Data Mean')

plt.title('Bayesian Updating: Beta-Binomial Conjugacy', fontsize=15)
plt.xlabel('Click-Through Rate (theta)', fontsize=12)
plt.ylabel('Density / Degree of Belief', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice how the Posterior is a compromise between the Prior and the Likelihood.")

## 5. The Effect of Prior Strength

The Bayesian framework acts as a natural regularizer. If we have a very strong prior belief (e.g., based on millions of past ad impressions), a small dataset will barely shift our belief. If we have a weak prior, the data easily overrides it.

Let us compare a Strong Prior vs a Weak Prior using the same data.

In [ ]:
# Weak Prior: Beta(1, 19) -> Mean 5%, but very low confidence (low sum of alpha+beta)
alpha_weak, beta_weak = 1, 19
alpha_weak_post = alpha_weak + observed_clicks
beta_weak_post = beta_weak + n_impressions - observed_clicks

# Strong Prior: Beta(50, 950) -> Mean 5%, but very high confidence (sum = 1000)
alpha_strong, beta_strong = 50, 950
alpha_strong_post = alpha_strong + observed_clicks
beta_strong_post = beta_strong + n_impressions - observed_clicks

# Compute PDFs
weak_post_pdf = stats.beta.pdf(x_ctr, alpha_weak_post, beta_weak_post)
strong_post_pdf = stats.beta.pdf(x_ctr, alpha_strong_post, beta_strong_post)

plt.figure(figsize=(10, 5))
plt.plot(x_ctr, weak_post_pdf, color='orange', linewidth=3, label='Posterior from Weak Prior')
plt.plot(x_ctr, strong_post_pdf, color='brown', linewidth=3, label='Posterior from Strong Prior')
plt.axvline(observed_clicks/n_impressions, color='blue', linestyle=':', label='Data Mean')

plt.title('Prior Sensitivity: Strong vs Weak Priors', fontsize=14)
plt.xlabel('Click-Through Rate (theta)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("The weak prior posterior centers near the data. The strong prior resists the new data entirely.")

## 6. Credible Intervals vs Confidence Intervals

A major advantage of Bayesian inference is the ability to construct Credible Intervals.

- Frequentist 95 percent Confidence Interval: 'If we ran this experiment 100 times, 95 of the intervals would contain the true parameter.' The parameter is fixed, the interval is random.
- Bayesian 95 percent Credible Interval: 'Given our data and prior, there is a 95 percent probability that the parameter lies within this range.' The interval is fixed, the parameter is random.

In [ ]:
# Calculate Bayesian 95% Credible Interval (Highest Density Interval equivalent for unimodal symmetric)
# We use the Percentile Point Function (PPF), which is the inverse of the CDF
lower_bound = stats.beta.ppf(0.025, alpha_post, beta_post)
upper_bound = stats.beta.ppf(0.975, alpha_post, beta_post)

print("--- Uncertainty Quantification ---")
print(f"Posterior Mean CTR: {alpha_post / (alpha_post + beta_post):.3f}")
print(f"95% Bayesian Credible Interval: [{lower_bound:.3f}, {upper_bound:.3f}]")
print("\nBusiness Interpretation: 'We are 95% certain that the true Click-Through Rate for this ad lies between " + 
      f"{lower_bound*100:.1f}% and {upper_bound*100:.1f}%.'")

## 7. Concept 3: The Intractable Denominator and MCMC

When likelihoods and priors are not cleanly conjugate (which is the case for almost all real-world machine learning models), we cannot use simple algebra. We must confront Bayes Theorem directly.

The denominator P(Data) requires integrating over the entire parameter space. For complex models, this integral is impossible to compute. We bypass this using Markov Chain Monte Carlo (MCMC).

MCMC Intuition: We build a 'robot' that wanders around the parameter space. The rules governing its movement ensure it spends time in different areas exactly proportional to the posterior probability. The histogram of its footsteps *is* the Posterior distribution.

In [ ]:
# Setup synthetic data for MCMC: Estimating the mean of a Normal distribution
true_mu = 7.5
n_samples = 40
continuous_data = np.random.normal(loc=true_mu, scale=2.0, size=n_samples)

print("--- Continuous Dataset for MCMC ---")
print(f"True Mean (mu): {true_mu}")
print(f"Sample Mean: {np.mean(continuous_data):.3f}")
print(f"Sample Size: {n_samples}")

## 8. Writing a Metropolis-Hastings Sampler from Scratch

The Metropolis-Hastings algorithm is the foundational MCMC method:
1. Start at a random parameter value (current_mu).
2. Propose a new step by adding random noise (proposed_mu).
3. Calculate the acceptance ratio: (Likelihood * Prior of proposed) / (Likelihood * Prior of current).
4. If ratio > 1, accept the move.
5. If ratio < 1, accept it probabilistically based on the ratio.

Let us implement this.

In [ ]:
# Define Prior: We believe mu is around 0 with standard deviation 5
def compute_prior(mu):
    return stats.norm.pdf(mu, loc=0, scale=5)

# Define Likelihood: P(Data | mu). Assuming known standard deviation of 2.0 for simplicity
def compute_likelihood(mu, data):
    # For numerical stability, production systems use Log-Likelihood.
    # We use simple multiplication here for educational clarity.
    probs = stats.norm.pdf(data, loc=mu, scale=2.0)
    return np.prod(probs)

# Define Unnormalized Posterior = Likelihood * Prior
def unnormalized_posterior(mu, data):
    return compute_likelihood(mu, data) * compute_prior(mu)

# Define the Metropolis-Hastings MCMC Sampler
def run_metropolis_hastings(data, iterations=10000, initial_mu=0.0):
    samples = []
    current_mu = initial_mu
    current_post = unnormalized_posterior(current_mu, data)
    accepted_moves = 0
    
    for _ in range(iterations):
        # Propose a new state via a random walk (Gaussian proposal)
        proposed_mu = np.random.normal(loc=current_mu, scale=0.5)
        proposed_post = unnormalized_posterior(proposed_mu, data)
        
        # Calculate acceptance ratio R
        if current_post == 0:
            acceptance_ratio = 1.0
        else:
            acceptance_ratio = proposed_post / current_post
            
        # Accept or Reject
        if acceptance_ratio >= 1.0 or np.random.rand() < acceptance_ratio:
            current_mu = proposed_mu
            current_post = proposed_post
            accepted_moves += 1
            
        samples.append(current_mu)
        
    acceptance_rate = accepted_moves / iterations
    return np.array(samples), acceptance_rate

## 9. Running the MCMC Sampler

We will run the sampler for 15,000 iterations. We discard the first 2,000 iterations as 'burn-in', because the chain takes some time to wander from its terrible initial guess (mu=0) to the high-probability region.

In [ ]:
n_iterations = 15000
burn_in = 2000

print("Starting MCMC Sampling...")
mcmc_samples, acc_rate = run_metropolis_hastings(continuous_data, iterations=n_iterations, initial_mu=0.0)

# Discard burn-in samples
valid_samples = mcmc_samples[burn_in:]

print(f"Sampling Complete.")
print(f"Acceptance Rate: {acc_rate:.2f} (Ideal rate is usually between 0.23 and 0.50)")
print(f"Total valid samples extracted: {len(valid_samples)}")

## 10. Visualizing MCMC Results (Trace and Posterior)

A Trace Plot shows the path the sampler took over time. A healthy trace plot looks like a 'hairy caterpillar', indicating the sampler is efficiently exploring the target distribution without getting stuck. The histogram of these samples forms our approximated Posterior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Trace Plot
axes[0].plot(range(len(valid_samples)), valid_samples, alpha=0.7, color='teal', linewidth=0.5)
axes[0].set_title("MCMC Trace Plot (Path of the Markov Chain)", fontsize=14)
axes[0].set_ylabel("Sampled Parameter Value (mu)")
axes[0].set_xlabel("Iteration (post burn-in)")
axes[0].grid(alpha=0.3)

# Posterior Histogram
axes[1].hist(valid_samples, bins=40, density=True, color='teal', alpha=0.7, edgecolor='black')
axes[1].axvline(true_mu, color='red', linestyle='--', linewidth=2, label=f"True mu ({true_mu})")
axes[1].axvline(np.mean(valid_samples), color='black', linewidth=2, label=f"Estimated mu ({np.mean(valid_samples):.2f})")
axes[1].set_title("Approximated Posterior Distribution", fontsize=14)
axes[1].set_xlabel("Parameter Value (mu)")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("The histogram of random walk samples perfectly reconstructs the probability distribution of the parameter!")

## 11. Practice Example: Bayesian A/B Testing

Unlike Frequentist A/B testing which requires fixed sample sizes and outputs confusing p-values, Bayesian A/B testing allows stakeholders to continuously monitor tests and directly answers: 'What is the probability Variant B is better than Variant A?'

In [ ]:
# Simulate A/B Test Data
visitors_A = 1200
conversions_A = 135  # ~11.2% true rate

visitors_B = 1250
conversions_B = 165  # ~13.2% true rate

print("--- A/B Test Raw Results ---")
print(f"Variant A: {conversions_A} conversions from {visitors_A} visitors ({(conversions_A/visitors_A)*100:.2f}%)")
print(f"Variant B: {conversions_B} conversions from {visitors_B} visitors ({(conversions_B/visitors_B)*100:.2f}%)")

### Evaluating A/B Test via Posterior Sampling

We assign a weak Beta(1,1) prior to both variants. We calculate their exact analytical posteriors. Then, we use Monte Carlo sampling to extract tens of thousands of potential CTRs from both distributions and compare them directly.

In [ ]:
# Uniform priors
prior_a, prior_b = 1, 1

# Posteriors
post_a_A, post_b_A = prior_a + conversions_A, prior_b + visitors_A - conversions_A
post_a_B, post_b_B = prior_a + conversions_B, prior_b + visitors_B - conversions_B

# Monte Carlo Sampling
n_ab_samples = 100000
samples_A = np.random.beta(post_a_A, post_b_A, n_ab_samples)
samples_B = np.random.beta(post_a_B, post_b_B, n_ab_samples)

# Direct probability calculation
prob_B_better = np.mean(samples_B > samples_A)
expected_uplift = np.mean((samples_B - samples_A) / samples_A)

print("--- Bayesian A/B Test Conclusion ---")
print(f"Probability that Variant B is better than Variant A: {prob_B_better * 100:.2f}%")
print(f"Expected relative uplift if we switch to Variant B:  {expected_uplift * 100:.2f}%")

### Visualizing A/B Test Posteriors

We can visualize the overlapping probability distributions. The area where Variant B's curve lies to the right of Variant A's curve represents our confidence in the winner.

In [ ]:
x_ab = np.linspace(0.08, 0.17, 500)
pdf_A = stats.beta.pdf(x_ab, post_a_A, post_b_A)
pdf_B = stats.beta.pdf(x_ab, post_a_B, post_b_B)

plt.figure(figsize=(10, 5))
plt.plot(x_ab, pdf_A, color='red', linewidth=2, label='Variant A Posterior')
plt.fill_between(x_ab, 0, pdf_A, color='red', alpha=0.3)

plt.plot(x_ab, pdf_B, color='blue', linewidth=2, label='Variant B Posterior')
plt.fill_between(x_ab, 0, pdf_B, color='blue', alpha=0.3)

plt.title('A/B Test Conversion Rate Posteriors', fontsize=14)
plt.xlabel('Conversion Rate (theta)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Thompson Sampling Intuition (Multi-Armed Bandit)

Bayesian inference naturally solves the Exploration vs Exploitation problem in reinforcement learning. In a Multi-Armed Bandit scenario with 3 slot machines, we maintain a Beta posterior for the payout rate of each machine.

Thompson Sampling Rule: At each step, draw one random sample from each posterior. Play the machine with the highest sampled value. This naturally explores uncertain machines and exploits proven winners.

In [ ]:
# Simulate 3 Bandits with different, unknown win rates
true_rates = [0.15, 0.25, 0.35]

# Initialize Priors (Beta(1,1) for all three)
alphas = np.array([1.0, 1.0, 1.0])
betas = np.array([1.0, 1.0, 1.0])

n_rounds = 1000
selections = []

for i in range(n_rounds):
    # 1. Sample from current posteriors
    samples = np.random.beta(alphas, betas)
    
    # 2. Pick the machine with the highest sample (Thompson Sampling)
    chosen_machine = np.argmax(samples)
    selections.append(chosen_machine)
    
    # 3. Simulate playing the machine and getting a reward
    reward = np.random.binomial(1, true_rates[chosen_machine])
    
    # 4. Update the posterior of the chosen machine
    alphas[chosen_machine] += reward
    betas[chosen_machine] += (1 - reward)

print("--- Multi-Armed Bandit Results ---")
for i in range(3):
    pulls = np.sum(np.array(selections) == i)
    print(f"Machine {i+1} (True Rate {true_rates[i]}): Pulled {pulls} times. Posterior Expected Rate: {alphas[i]/(alphas[i]+betas[i]):.3f}")

print("\nNotice how the algorithm quickly learned to almost exclusively exploit the best machine (Machine 3).")

## 13. Maximum A Posteriori (MAP) vs Full Bayesian Inference

Full Bayesian inference gives you a complete distribution. Maximum A Posteriori (MAP) estimation takes a shortcut: it simply finds the single peak (mode) of the posterior distribution using standard optimization (like Gradient Descent).

MAP bridges Bayesian stats with frequentist machine learning. Adding a Gaussian prior and finding the MAP is mathematically identical to performing L2 (Ridge) Regularization.

In [ ]:
from scipy.optimize import minimize

# We define the Negative Unnormalized Log-Posterior to minimize it
def negative_log_posterior(mu, data):
    # Log Likelihood
    log_lik = np.sum(stats.norm.logpdf(data, loc=mu, scale=2.0))
    # Log Prior (Normal distribution centered at 0)
    log_prior = stats.norm.logpdf(mu, loc=0, scale=5)
    return -(log_lik + log_prior)

# Run optimization to find MAP
opt_result = minimize(negative_log_posterior, x0=0.0, args=(continuous_data,))
map_estimate = opt_result.x[0]

print("--- MAP vs Full MCMC Expected Value ---")
print(f"MAP Estimate (The Peak):      {map_estimate:.3f}")
print(f"MCMC Posterior Mean Estimate: {np.mean(valid_samples):.3f}")
print("\nFor symmetric posteriors like Gaussian, MAP and Posterior Mean are identical. However, MAP provides no uncertainty quantification!")

## 14. Summary and Key Takeaways

- **Probability as Belief**: Bayesian Inference treats parameters as random variables described by distributions, updating them as new data arrives.
- **The Engine**: Posterior is proportional to Likelihood * Prior.
- **Conjugate Priors**: Allow for exact, closed-form algebraic solutions by ensuring the prior and posterior share the same distribution family (e.g., Beta-Binomial).
- **MCMC**: Markov Chain Monte Carlo bypasses intractable mathematical integrals by simulating a random walk that converges to the posterior distribution.
- **Actionable Uncertainty**: Bayesian models produce Credible Intervals, directly answering questions like 'What is the probability this parameter is between A and B?', which frequentist methods cannot do.

## 15. Common Pitfalls to Avoid

> 1. **Cromwell's Rule (Zero Probability Priors)**: If you assign a Prior probability of exactly 0 to an event, the Posterior will *always* be 0, regardless of evidence. Never use strictly bounded uniform priors unless physical reality dictates it.
> 
> 2. **Ignoring MCMC Convergence**: Running MCMC is not a black box. You must check convergence metrics (like R-hat / Gelman-Rubin statistic) to ensure different random chains converge to the same distribution. If the trace plot doesn't look like a 'hairy caterpillar', your results are invalid.
> 
> 3. **Mixing CI Definitions**: Do not explain a Frequentist Confidence Interval using Bayesian terminology. Only a Bayesian Credible Interval represents the probability of the parameter being in a specific range.

## 16. Interview Guide and Concept Review

**Q: How does a Bayesian approach handle small datasets compared to a Frequentist approach?**
A: A Frequentist approach on small datasets often yields massive, uninformative Confidence Intervals or point estimates highly susceptible to outliers. A Bayesian approach anchors the estimate using a Prior. The Prior acts as a regularizer, preventing extreme conclusions when evidence is sparse.

**Q: What is the difference between MAP and fully Bayesian Inference?**
A: MAP (Maximum A Posteriori) finds the single peak (mode) of the posterior distribution. It is a point estimate that incorporates a prior (equivalent to Regularization). Fully Bayesian inference calculates the entire posterior distribution, allowing for true uncertainty quantification.

**Q: Why don't we use Bayesian methods for everything if they are so intuitive?**
A: Computation cost. Maximum Likelihood Estimation is often just a fast derivative optimization (Gradient Descent). Fully Bayesian inference requires calculating intractable integrals over complex spaces, often requiring thousands of MCMC simulation steps, which scales poorly to massive datasets.